<a href="https://colab.research.google.com/github/yuvrajrajput/transformer-from-scratch/blob/main/Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Project Architecture

To build a production-level system, a well-organized project structure is crucial. We will adhere to a modular design to keep the codebase clean, scalable, and easy to maintain. Here's the proposed directory structure and its purpose:

```
nmt_sanskrit/
├── data/                   # Contains raw, processed, and tokenized datasets
│   ├── raw/                # Original datasets (e.g., parallel corpora, vocabulary files)
│   └── processed/          # Cleaned, tokenized, and preprocessed data ready for training
├── models/                 # Stores trained model checkpoints and configurations
│   ├── checkpoints/        # Saved model states during training
│   └── pretrained/         # Optionally, pre-trained embeddings or models
├── training/               # Scripts and utilities for model training and evaluation
│   ├── __init__.py
│   ├── trainer.py          # Main training loop logic
│   ├── optimizers.py       # Custom optimizers or learning rate schedulers
│   └── evaluators.py       # Metrics calculation (BLEU, token accuracy, etc.)
├── inference/              # Scripts and utilities for model inference
│   ├── __init__.py
│   ├── predictor.py        # Logic for making predictions (greedy, beam search)
│   └── api_example.py      # Example script for deploying an inference API
├── transformer/            # Core Transformer architecture implementation
│   ├── __init__.py
│   ├── attention.py        # Scaled Dot Product, Multi-Head Attention
│   ├── embeddings.py       # Input Embeddings, Positional Encoding
│   ├── layers.py           # Feed Forward Network, Layer Normalization, Residual Connections
│   └── transformer.py      # Encoder, Decoder, and full Transformer model assembly
├── tokenizer/              # Custom tokenizer implementation and vocabulary management
│   ├── __init__.py
│   ├── custom_tokenizer.py # Logic for building and using our custom tokenizer
│   └── vocab_builder.py    # Utilities for vocabulary creation
├── utils/                  # General utility functions (logging, config management, GPU setup)
│   ├── __init__.py
│   ├── config.py           # Configuration parsing (e.g., using OmegaConf or simple dicts)
│   ├── helpers.py          # Miscellaneous helper functions
│   └── logger.py           # Logging setup
├── config.yaml             # Main configuration file for the entire project
├── README.md               # Project documentation
└── requirements.txt        # List of all Python dependencies
```

For a Colab environment, we'll simulate this structure by creating directories and saving files within them, or by placing related code in distinct cells for clarity. Each major component will be explained in detail.

## 2. Dependency Installation

First, we need to install the necessary libraries. PyTorch will be our deep learning framework, and `sentencepiece` will be used for tokenization due to its efficiency and multilingual capabilities, especially for Sanskrit. We'll also need `torchtext` for utilities, and `einops` for clearer tensor manipulation.

In [1]:
# Uninstall existing versions to ensure a clean install
!pip uninstall -y torch torchvision torchaudio sentencepiece einops torchtext

# Install specific compatible versions of PyTorch and torchtext for CUDA 11.8
# torch 2.2.2 is compatible with torchtext 0.17.2
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118
!pip install sentencepiece
!pip install einops
!pip install torchtext==0.17.2

import torch
import torch.nn as nn
import torch.optim as optim
import math
import sentencepiece as spm
from torch.utils.data import Dataset, DataLoader
from torchtext.data.utils import get_tokenizer
from collections import Counter
from einops import rearrange, repeat

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available. Running on CPU.")

# Create project directories
import os

def create_directories(base_path='.'):
    dirs = [
        'data/raw',
        'data/processed',
        'models/checkpoints',
        'training',
        'inference',
        'transformer',
        'tokenizer',
        'utils'
    ]
    for d in dirs:
        os.makedirs(os.path.join(base_path, d), exist_ok=True)
        # Create an __init__.py for Python packages
        if os.path.basename(d) in ['training', 'inference', 'transformer', 'tokenizer', 'utils']:
            with open(os.path.join(base_path, d, '__init__.py'), 'w') as f:
                pass

create_directories()
print("Project directories created.")

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: einops 0.8.2
Uninstalling einops-0.8.2:
  Successfully uninstalled einops-0.8.2
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 113.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

PyTorch version: 2.2.2+cu118
CUDA available: True
CUDA device name: Tesla T4
Project directories created.


## 3. Dataset Preparation

We'll start with a small, toy English-Sanskrit parallel dataset. This helps us quickly test our Transformer implementation before scaling up. We will then discuss how to handle larger datasets.

### Toy Dataset

Our toy dataset will consist of simple English sentences and their corresponding Sanskrit translations. This will be a small list of pairs.

### Scaling Datasets

For a real-world scenario, you would typically use large parallel corpora. Some sources for English-Sanskrit data include:
*   **IndicCorp:** A large collection of Indian language corpora, often including parallel texts.
*   **Online resources:** Various academic projects or digital libraries might host parallel texts.
*   **Translation memories:** If available, these can be a rich source.

When scaling, key considerations are:
*   **Data Volume:** The more data, the better the model performs (generally).
*   **Data Quality:** Clean data is paramount. Noise, misalignments, and errors can significantly degrade model performance.
*   **Domain Relevance:** Data from the target domain will lead to better performance for that specific domain.

### Data Cleaning and Normalization

Before tokenization, raw text data often requires cleaning and normalization:
1.  **Remove HTML tags or special characters:** If data is scraped from the web.
2.  **Normalize Unicode:** Ensure consistent representation of characters (e.g., using `unicodedata.normalize`).
3.  **Lowercase:** Often done for English, but might be tricky for Sanskrit where case is not a concept, but transliteration differences exist.
4.  **Remove extra whitespace:** Consolidate multiple spaces into one.
5.  **Handle punctuation:** Standardize or separate punctuation from words.
6.  **Filter short/long sentences:** Remove sentences that are too short (often noise) or too long (computationally expensive).
7.  **Remove duplicates:** Ensure unique sentence pairs.

### Sanskrit Tokenization Challenges

Sanskrit presents unique challenges for tokenization compared to English:
*   **Sandhi:** Sanskrit words often combine through a process called Sandhi, where adjacent sounds merge. This makes simple whitespace tokenization insufficient. A single 'word' might represent multiple underlying morphemes.
*   **Compounding:** Sanskrit has a rich tradition of compounding words, forming very long compound nouns or verbs. Identifying individual components requires linguistic knowledge.
*   **Inflectional Morphology:** Words are highly inflected for case, number, gender, tense, mood, etc. A root word can have many forms.
*   **Script:** Devanagari script (and others) can be complex to process without proper Unicode handling.

`SentencePiece` (or BPE/WordPiece) handles many of these challenges by learning subword units, which is crucial for languages like Sanskrit where the vocabulary can be extremely large due to inflection and compounding. It does not rely on pre-defined word boundaries and can segment words into smaller, frequently occurring units.

### Vocabulary Building

After tokenization, a vocabulary needs to be built. This maps unique tokens (or subword units) to numerical IDs. Key aspects:
*   **Frequency-based:** Tokens are typically ordered by frequency, and a fixed vocabulary size is chosen.
*   **Special Tokens:** We will need special tokens like:
    *   `<PAD>`: Padding token, used to make all sequences in a batch the same length.
    *   `<UNK>`: Unknown token, for words not in the vocabulary.
    *   `<BOS>`: Beginning-of-Sentence token, marks the start of a sequence.
    *   `<EOS>`: End-of-Sentence token, marks the end of a sequence.
*   **Vocabulary Size:** A balance between covering most words and keeping the model size manageable. Subword tokenization helps keep this size reasonable.

In [ ]:
# Create a toy English-Sanskrit parallel dataset
raw_data = [
    ("I eat food.", "अहं भोजनं खादामि।"),
    ("You drink water.", "त्वं जलं पिबसि।"),
    ("He reads a book.", "सः पुस्तकं पठति।"),
    ("She writes a letter.", "सा पत्रं लिखति।"),
    ("We go home.", "वयं गृहं गच्छामः।"),
    ("They play outside.", "ते बहिः क्रीडन्ति।"),
    ("The sun is bright.", "सूर्यः तेजस्वी अस्ति।"),
    ("The moon is beautiful.", "चन्द्रः सुन्दरः अस्ति।"),
    ("What is your name?", "तव नाम किम् अस्ति?"),
    ("How are you?", "कथम् अस्ति भवान्/भवती?")
]

# Save the raw data to files
with open('data/raw/eng.txt', 'w', encoding='utf-8') as f_eng,
     open('data/raw/san.txt', 'w', encoding='utf-8') as f_san:
    for eng_sentence, san_sentence in raw_data:
        f_eng.write(eng_sentence + '\n')
        f_san.write(san_sentence + '\n')

print("Toy dataset saved to data/raw/eng.txt and data/raw/san.txt")

## 4. Tokenizer Implementation

Tokenization is a critical step in any NLP pipeline, especially for Neural Machine Translation. For Sanskrit, traditional word-based tokenization is challenging due to phenomena like Sandhi and compounding. Subword tokenization, such as that provided by SentencePiece, offers a robust solution by learning common subword units, thus handling rare words and morphological variations effectively.

### SentencePiece Tokenizer

SentencePiece is a language-agnostic subword tokenizer. It treats the input as a raw stream of Unicode characters and generates a vocabulary of subword units directly from the raw text. It supports two main algorithms: BPE (Byte Pair Encoding) and unigram. For NMT, both are effective.

**Why SentencePiece?**
*   **Handles OOV words:** By breaking words into subword units, it can represent rare or unseen words.
*   **Reduces vocabulary size:** Compared to word-level tokenization, it keeps the vocabulary size manageable while still representing a vast number of words.
*   **Language Agnostic:** Works well across various languages, including those with complex morphology like Sanskrit.
*   **End-to-End Training:** No need for pre-segmentation; it works directly on raw text.

### Special Tokens

For NMT, we need several special tokens:
*   `<PAD>` (Padding): Used to ensure all sequences in a batch have the same length. Typically assigned ID 0.
*   `<UNK>` (Unknown): Represents tokens not in the vocabulary. SentencePiece often handles this inherently by breaking unknown words into known subwords, but explicitly having one can be useful.
*   `<BOS>` (Beginning of Sentence): Marks the start of a target sequence (decoder input).
*   `<EOS>` (End of Sentence): Marks the end of a sequence, used to signal the decoder to stop generating tokens.

SentencePiece automatically adds an `<unk>` token. We'll explicitly add `<pad>`, `<bos>`, and `<eos>` tokens during training.

### Training the SentencePiece Model

We will train separate SentencePiece models for English and Sanskrit. This allows each tokenizer to learn the optimal subword units for its respective language. The vocabulary size (`vocab_size`) is a hyperparameter that balances vocabulary coverage and model size.

In [ ]:
# Define file paths for the raw data
eng_file = 'data/raw/eng.txt'
san_file = 'data/raw/san.txt'

# Define model prefixes and vocabulary sizes
eng_model_prefix = 'tokenizer/eng_spm'
san_model_prefix = 'tokenizer/san_spm'
vocab_size = 5000 # A reasonable starting point, can be tuned

# Train SentencePiece model for English
# The `user_defined_symbols` ensures our special tokens are always in the vocabulary.
# `character_coverage` ensures a certain percentage of characters are covered by the vocabulary.
# `model_type=unigram` or `bpe` can be chosen. Unigram often performs well.
print("Training English SentencePiece model...")
spm.SentencePieceTrainer.train(
    f'--input={eng_file} --model_prefix={eng_model_prefix} --vocab_size={vocab_size} \
    --model_type=unigram --character_coverage=0.9995 --bos_id=-1 --eos_id=-1 --unk_id=0 \
    --pad_id=1 --user_defined_symbols=<bos>,<eos>,<pad>'
)
print(f"English SentencePiece model trained and saved to {eng_model_prefix}.model and {eng_model_prefix}.vocab")

# Train SentencePiece model for Sanskrit
print("Training Sanskrit SentencePiece model...")
spm.SentencePieceTrainer.train(
    f'--input={san_file} --model_prefix={san_model_prefix} --vocab_size={vocab_size} \
    --model_type=unigram --character_coverage=0.9995 --bos_id=-1 --eos_id=-1 --unk_id=0 \
    --pad_id=1 --user_defined_symbols=<bos>,<eos>,<pad>'
)
print(f"Sanskrit SentencePiece model trained and saved to {san_model_prefix}.model and {san_model_prefix}.vocab")

# Load the trained tokenizers
eng_tokenizer = spm.SentencePieceProcessor()
eng_tokenizer.load(f'{eng_model_prefix}.model')
san_tokenizer = spm.SentencePieceProcessor()
san_tokenizer.load(f'{san_model_prefix}.model')

# Verify special tokens and their IDs
# SentencePiece assigns IDs based on training. We need to map them consistently.
# Note: `bos_id`, `eos_id`, `pad_id` are *not* automatically assigned to our user_defined_symbols
# if we set them to -1. We need to retrieve them or assign them after training.

# Let's get the special token IDs. SentencePiece's `unk_id` is usually 0 by default if not set.
# We explicitly set `unk_id=0`, `pad_id=1` during training, so these are fixed.
UNK_ID = eng_tokenizer.unk_id()
PAD_ID = eng_tokenizer.pad_id() # Should be 1 as specified
BOS_ID = eng_tokenizer.piece_to_id('<bos>')
EOS_ID = eng_tokenizer.piece_to_id('<eos>')

# For demonstration, let's tokenize a sentence and print IDs
eng_sentence = raw_data[0][0]
san_sentence = raw_data[0][1]

print(f"\nOriginal English: {eng_sentence}")
print(f"Tokenized English (pieces): {eng_tokenizer.encode_as_pieces(eng_sentence)}")
print(f"Tokenized English (IDs): {eng_tokenizer.encode_as_ids(eng_sentence)}")

# To include BOS/EOS, we'll manually add them during data processing
eng_ids_with_special = [BOS_ID] + eng_tokenizer.encode_as_ids(eng_sentence) + [EOS_ID]
print(f"Tokenized English with BOS/EOS (IDs): {eng_ids_with_special}")

print(f"\nOriginal Sanskrit: {san_sentence}")
print(f"Tokenized Sanskrit (pieces): {san_tokenizer.encode_as_pieces(san_sentence)}")
print(f"Tokenized Sanskrit (IDs): {san_tokenizer.encode_as_ids(san_sentence)}")

san_ids_with_special = [BOS_ID] + san_tokenizer.encode_as_ids(san_sentence) + [EOS_ID]
print(f"Tokenized Sanskrit with BOS/EOS (IDs): {san_ids_with_special}")

print(f"\nSpecial Tokens IDs (English/Sanskrit, should be same):")
print(f"<UNK> ID: {UNK_ID}")
print(f"<PAD> ID: {PAD_ID}")
print(f"<BOS> ID: {BOS_ID}")
print(f"<EOS> ID: {EOS_ID}")

# Store tokenizers and special IDs for later use
import json
with open('tokenizer/tokenizers.json', 'w', encoding='utf-8') as f:
    json.dump({
        'eng_model_path': f'{eng_model_prefix}.model',
        'san_model_path': f'{san_model_prefix}.model',
        'unk_id': UNK_ID,
        'pad_id': PAD_ID,
        'bos_id': BOS_ID,
        'eos_id': EOS_ID,
        'eng_vocab_size': eng_tokenizer.get_piece_size(),
        'san_vocab_size': san_tokenizer.get_piece_size()
    }, f, indent=4)
print("Tokenizer paths and special IDs saved to tokenizer/tokenizers.json")